In [1]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: d:\Programing\CyberSec-Reasoner


In [63]:
import re
import random
import warnings
from pprint import pprint
from datasets import load_dataset, Dataset, DatasetDict

warnings.filterwarnings("ignore")

*****
# CVE to CWE Mapping Evaluation Dataset for GRPO trained model
Huggingface Dataset Link: [link](https://huggingface.co/datasets/stasvinokur/cve-and-cwe-dataset-1999-2025)

In [61]:
ds = load_dataset("stasvinokur/cve-and-cwe-dataset-1999-2025", split="train", streaming=True)
ds = iter(ds)

In [64]:
sample = next(ds)
pprint(sample)
if re.match(r"CVE-\d{4}-\d{4,7}", sample["CVE-ID"]):
    print(f"CWE-ID: {sample['CWE-ID']}")

{'CVE-ID': 'CVE-2004-1849',
 'CVSS-V2': 4.3,
 'CVSS-V3': None,
 'CVSS-V4': None,
 'CWE-ID': 'NVD-CWE-Other',
 'DESCRIPTION': 'Multiple cross-site scripting (XSS) vulnerabilities in cPanel '
                '9.1.0 allow remote attackers to inject arbitrary web script '
                'or HTML via the (1) email parameter to dodelautores.html or '
                '(2) handle parameter to addhandle.html.',
 'ID': 9898,
 'SEVERITY': 'MEDIUM'}
CWE-ID: NVD-CWE-Other


In [51]:
system_prompt = """You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.

Your task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).

You must follow a structured reasoning workflow:

1. Understand the vulnerability context and affected components  
2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  
3. Determine the root cause of the weakness  
4. Map the root cause to the most appropriate CWE category  

Guidelines:
- Focus on root cause rather than surface-level keywords  
- Use precise cybersecurity terminology  
- Ensure reasoning clearly supports the final CWE selection  
- Prefer the most specific applicable CWE when possible  

Avoid:
- Guessing without justification  
- Contradictions between reasoning and conclusion  
- Irrelevant or overly generic explanations  
- Blindly copying CWE identifiers from the input  

Output Format (STRICT):

1. First, provide detailed step-by-step reasoning inside:
   <think> ... </think>

2. Then provide a structured analytical explanation (clear, well-organized, 1–2 paragraphs) that summarizes:
   - the vulnerability type  
   - the root cause  
   - why the selected CWE is the best match  

3. The last line must contain ONLY the CWE ID (e.g., CWE-79)

Important:
- The explanation must be consistent with the reasoning  
- The CWE must be fully justified by both reasoning and explanation  
- Do not include anything after the CWE ID  
"""

In [65]:
prompt = ("Analyze the following CVE description and map it to the appropriate CWE. "
        "Provide a brief justification for your choice. Ensure the last line of your " 
        "response contains only the CWE ID. CVE Description:")
dataset = []
counter = 0
for i, item in enumerate(iter(ds)):
    if re.match(r"^CWE-\d+$", item["CWE-ID"]):
        dataset.append({
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"{prompt} {item['DESCRIPTION']}"},
                {"role": "assistant", "content": item["CWE-ID"]}
            ]
        })
        counter += 1
    if counter >= 1000:
        break
dataset = DatasetDict({
    "test": Dataset.from_list(dataset)
})
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['messages'],
        num_rows: 1000
    })
})


In [75]:
dataset['test'][random.randint(0, len(dataset['test']) - 1)]

{'messages': [{'content': 'You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.\n\nYour task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).\n\nYou must follow a structured reasoning workflow:\n\n1. Understand the vulnerability context and affected components  \n2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  \n3. Determine the root cause of the weakness  \n4. Map the root cause to the most appropriate CWE category  \n\nGuidelines:\n- Focus on root cause rather than surface-level keywords  \n- Use precise cybersecurity terminology  \n- Ensure reasoning clearly supports the final CWE selection  \n- Prefer the most specific applicable CWE when possible  \n\nAvoid:\n- Guessing without justification  \n- Contradictions between reasoning and conclusion  \n- Irrelevant or overly generic explanations  \n- Blindly co

In [76]:
dataset.save_to_disk("data/processed/grpo_evaluation_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 6658.70 examples/s]
